## 4. Object-Oriented Programming (OOP)

**Object-Oriented Programming (OOP)** is a paradigm that organizes code around **objects** — bundles of state (data) and behavior (the functions that operate on that data) — instead of around a plain sequence of instructions or a set of standalone functions passing separate data structures around. 

Classes provide a means of **bundling data and functionality** together. Creating a new class creates a new type of object, allowing new instances of that type to be made.

Almost everything in Python is already an object: an `int`, a `list`, a function, even a class itself (a class is an instance of `type`). Writing your own classes just means extending this same system with your own custom object types, tailored to your problem.

OOP is traditionally explained through **four concepts**. This section is organized around them, building up from the mechanics of defining a class to how objects relate to and interact with each other.

1. **Encapsulation** — bundling data and the methods that operate on it together, and controlling access to that data.
2. **Abstraction** — exposing only what's necessary to use an object, hiding the implementation details behind a simpler interface.
3. **Inheritance** — building new classes on top of existing ones, reusing and extending their behavior.
4. **Polymorphism** — writing code that works with objects of different types, as long as they support the same interface.

### 4.1 Class Fundamentals

A **class** is a blueprint: a template describing what data **(attributes)** and behavior **(methods)** its objects will have, without holding any of that data itself. An **instance** (or **object**) is a concrete value built from that class, with its own data/independent state.

Each class instance can have attributes attached to it for maintaining its state. Class instances can also have methods (defined by its class) for modifying its state.

Analogy: a class is like a house blueprint — it says "there will be a kitchen here, a bedroom there." An instance is an actual house built from it. You can build many houses (instances) from one blueprint (class), each furnished differently (different attribute values), but all sharing the same layout (the same methods).

A class is defined with the `class` keyword (by convention, `PascalCase` names):

```python
class ClassName:
    ...
```

You define the properties all `Class` objects must have with the `.__init__()` method. `__init__` is the **initializer**. It is *not* technically the constructor — by the time `__init__` runs, the object already exists in memory (`__new__` is the actual constructor, which is rarely needed to touch). `__init__`'s job is simply to set up the object's initial state. Python calls it automatically right after creating the object, whenever you write `ClassName(arguments)`. `__init__` initializes each new instance of a class.

`self` is the first parameter of every instance method, by convention named `self`. It refers to the specific instance the method is being called on. Python passes it automatically: `dog.bark()` is really shorthand for `Dog.bark(dog)`. `self` is not a keyword — it's pure convention — but you should never deviate from it; every Python developer expects it.

Creating a new object from a class is called **instantiating** a class. You can create a new object by typing the name of the class, followed by opening and closing parentheses: `obj = Class(...)`.

In [ ]:
class Dog:
    """A simple class representing a dog."""

    species = "Canis familiaris"  # class attribute: one copy, shared by every Dog

    def __init__(self, name, breed):
        self.name = name    # instance attribute: this Dog's own data
        self.breed = breed  # instance attribute

    def bark(self): # instance method
        return f"{self.name} says Woof!"

rex = Dog("Rex", "Labrador")
buddy = Dog("Buddy", "Poodle")

print(rex.bark())                              # Dog.bark(rex) under the hood
print(rex.name, rex.breed, rex.species)
print(buddy.name, buddy.breed, buddy.species)
print(type(rex), isinstance(rex, Dog))

**Instance attributes** (set with `self.attr = ...`, usually inside `__init__`) belong to one single object, are unique to each instance. **Class attributes** (assigned directly in the class body, outside any method) live once on the class itself and are shared by every instance.

An instance attribute’s value is specific to a particular instance of the class. Class attributes are attributes that have the same value for all class instances.

**Instance methods** are functions that you define **inside** a class and can only call on an instance of that class. Just like `.__init__()`, an instance method always takes `self` as its first parameter. A method is a function that belongs to an object.

Attribute lookup order: when you write `obj.attr`, Python first checks the instance's own `__dict__`; only if it's not found there does it check the class (and then the class's parents). This has two important consequences:

* **Reading** a class attribute through an instance (`rex.species`) works fine — it's found on the class.
* **Assigning** to it through an instance (`rex.species = "X"`) does *not* modify the class attribute. It creates a brand-new *instance* attribute that only shadows the class attribute for that one object — every other instance is unaffected.

In [ ]:
rex.species = "Canis lupus"   # creates a NEW instance attribute, doesn't touch the class attribute
print(rex.species, buddy.species, Dog.species)

Dog.species = "Renamed"       # THIS actually mutates the class attribute
print(rex.species, buddy.species)  # rex still shows its own shadowed value; buddy follows the class

Reassignment shadows, but **mutating a mutable class attribute in place** (`.append()`, `.update()`, ...) *does* affect every instance, because there's still only one shared list/dict object behind the scenes. This is one of the most common OOP bugs for beginners — it's the exact same trap as the mutable-default-argument pitfall from the Functions section.

In [ ]:
class Team:
    members = []  # DANGER: one list object, shared by every Team instance

    def __init__(self, name):
        self.name = name

    def add_member(self, person):
        self.members.append(person)  # mutates the SHARED class attribute in place

alpha = Team("Alpha")
beta = Team("Beta")

alpha.add_member("Ana")
print(alpha.members)  # ['Ana']
print(beta.members)   # ['Ana'] too! Both teams were secretly sharing the same list


class FixedTeam:
    def __init__(self, name):
        self.name = name
        self.members = []  # instance attribute: a fresh list per Team

    def add_member(self, person):
        self.members.append(person)

alpha2, beta2 = FixedTeam("Alpha"), FixedTeam("Beta")
alpha2.add_member("Ana")
print(alpha2.members, beta2.members)  # correctly independent

#### **The `.__dict__` Attribute**

In Python, both classes and instances have a special attribute called `.__dict__`. This attribute contains a **dictionary** containing the writable members/names of the underlying class or instance.

In a class, `.__dict__` will contain **class attributes and methods.** In an instance, `.__dict__` will contain **instance attributes.**

In [ ]:
class MyDict:
    x = 0
    def __init__(self, a, b, c):
        self.a = a
        self.b = b
        self.c = c 

    def printer(self):
        return f"Class {type(self).__name__}: {self.a}, {self.b} and {self.c}"

test = MyDict(1, 2, 3)
print(test.__dict__)
print(MyDict.__dict__)

### 4.2 Encapsulation & Method Types

**Encapsulation** means two things: **bundling** data and the behavior that operates on it into one unit (already achieved just by using a class), and **controlling** how that data is accessed from outside.

#### **Naming Conventions**

Python's philosophy here is famously summarized as "we're all consenting adults here": there is no enforced `private` keyword like in Java or C++. Python doesn’t distinguish between private, protected, or public attributes. All attributes are accessible in one way or another. Access control is done purely by **naming convention** — a social contract between you and whoever reads your code (including future you):

* `name` — **public**. Part of the intended interface, free to use from anywhere.
* `_name` — **non-public/protected by convention**. A single leading underscore signals "internal use, not part of the public API." Python does nothing to stop external access; it's a request, not a lock.
* `__name` — **name-mangled**. Two leading underscores (and at most one trailing underscore) cause Python to automatically rewrite every occurrence of `__name` inside the class body to `_ClassName__name`. This is called **name mangling**, and its real purpose is *not* primarily privacy — it exists mainly to prevent accidental attribute clashes when subclasses define an attribute with the same name.

Public names are part of the official interface of your classes, while non-public names aren’t intended to be. This last only indicates that the attribute isn’t intended to be used directly from outside the containing class.

Name mangling is an automatic name transformation that prepends the class’s name to the member’s name, like in `_ClassName__attribute` or `_ClassName__method`. This transformation results in name hiding. In other words, mangled names aren’t available for direct access and aren’t part of a class’s public API.

In [ ]:
class Widget:
    def __init__(self, value):
        self._protected = value      # convention: "internal use", still fully accessible
        self.__private = value * 2   # name-mangled to _Widget__private

    def show_private(self):
        return self.__private        # works fine from inside the class body

w = Widget(10)
print(w._protected)          # allowed, just a convention (linters may warn about it)
print(w.show_private())      # 20

print(vars(w))                # real keys are '_protected' and '_Widget__private'
print(w._Widget__private)     # still reachable if you know the mangled name: obfuscation, not real privacy

try:
    print(w.__private)        # mangled at the SOURCE-CODE level, so this literal name doesn't exist on w
except AttributeError as e:
    print("Rejected:", e)

#### **Property and Descriptor-Based Attributes**

Python allows you to add function-like behavior on top of existing instance attributes and turn them into **managed attributes.** You can have function-like behavior and attribute-like access at the same time.

Python provides handy tools that allow you to change the underlying implementation of your attributes without changing your public API. The best approach is to turn your attributes into **properties.**

To create a managed attribute with function-like behavior in Python, you can use either a property or a descriptor.

##### **Properties:**

Properties represent an intermediate functionality between a plain attribute, or field, and a method. In other words, they allow you to create methods that behave like attributes.

`property()` is a **built-in** function that allows you to turn class attributes into **properties** or **managed attributes.** With `property()`, you can attach implicit getter and setter methods to given class attributes:

```python
property([fget=None, fset=None, fdel=None, doc=None])
```
The first two arguments take function objects that will play the role of getter (`fget`) and setter (`fset`) methods. Here is what each argument does:

| Argument | Description |
|---|---|
|`fget`| A function object that returns the value of the managed attribute. |
| `fset` | A function object that allows you to set the value of the managed attribute. |
| `fdel` | A function object that defines how the managed attribute handles deletion. |
| `doc` | Property's docstring. |

<br>

You can use `property()` either as a **function** or **decorator** to build your properties.

* **`property()`:** In the end, properties are class attributes that manage instance attributes:

In [ ]:
class Circle:
    def __init__(self, radius):
        self._radius = radius

    def _get_radius(self):
        print("Get radius")
        return self._radius

    def _set_radius(self, value):
        print("Set radius")
        self._radius = value

    def _del_radius(self):
        print("Delete radius")
        del self._radius

    radius = property(         # Class attribute to store the property object. Now hides the non-public
        fget=_get_radius,      # attribute ._radius (now managed attribute). Access and assign .radius directly
        fset=_set_radius,
        fdel=_del_radius,
        doc="The radius property."
    )

* **`@property`:** 

In [ ]:
class Circle:
    def __init__(self, radius):
        self._radius = radius

    @property
    def radius(self): # Getter method
        """The radius property."""
        print("Get radius")
        return self._radius

    @radius.setter 
    def radius(self, value): # Setter method
        print("Set radius")
        self._radius = value

    @radius.deleter
    def radius(self): # Deleter method
        print("Delete radius")
        del self._radius

**`@property`** solves a common evolution problem: you start with a plain public attribute; later you need validation, a computed value, or logging whenever it's read or written — but changing `obj.attr` into `obj.get_attr()`/`obj.set_attr()` would break every existing caller.

`@property` turns a method into something accessed *without* parentheses, like a plain attribute. `@<name>.setter` lets you intercept assignment, and `@<name>.deleter` lets you intercept `del obj.<name>`. The usual pattern is to store the real value in a protected attribute (`_x`) and expose it through a property named `x`, so validation logic runs on every read/write without callers ever noticing the difference. Defining only a getter (no setter) creates a **read-only** property — any assignment attempt raises `AttributeError`.

In [ ]:
class Account:
    def __init__(self, balance):
        self._balance = balance   # the "real" storage

    @property
    def balance(self):
        """Getter: reads look like a plain attribute access."""
        return self._balance

    @balance.setter
    def balance(self, value):
        if value < 0:
            raise ValueError("Balance can't be negative")
        self._balance = value

    @balance.deleter
    def balance(self):
        print("Closing account, clearing balance")
        self._balance = 0

acc = Account(100)
print(acc.balance)     # calls the getter, no ()
acc.balance = 150        # calls the setter: validated assignment
print(acc.balance)

try:
    acc.balance = -10
except ValueError as e:
    print("Rejected:", e)

del acc.balance          # calls the deleter
print(acc.balance)

In general, you should avoid using properties for attributes that don’t require extra functionality or processing.

##### - **Providing Read-Only Attributes**

Probably the most elementary use case of `property()` is to provide **read-only attributes** in your classes. Say you need an immutable class that does not allow the user to mutate the original attributes values:

In [ ]:
class Point:
    def __init__(self, x, y):
        self._x = x
        self._y = y

    @property
    def x(self):
        return self._x

    @property
    def y(self):
        return self._y

point = Point(1, 1)
# Read
point.x
point.y

point.x = 2 # -> AttributeError

##### **- Creating Read-Write Attributes**

You can also use `property()` to provide managed attributes with **read-write** capabilities.

In [ ]:
class Circle:
    def __init__(self, radius):
        self.radius = radius # Initializer does not assign to a non-public attribute but to the 
                             # .radius property to ensure every radius (inc. initial) goes through setter
    @property
    def radius(self):
        return self._radius

    @radius.setter
    def radius(self, value):
        self._radius = float(value)

    @property
    def diameter(self): # Getter
        return self.radius * 2

    @diameter.setter # Setter
    def diameter(self, value):
        self.radius = value / 2

##### **- Providing Write-Only Attributes**

You can also use `property()` to create **write-only** attributes by tweaking the getter method of properties.

In [ ]:
import hashlib
import os

class User:
    def __init__(self, name, password):
        self.name = name
        self.password = password

    @property
    def password(self):
        raise AttributeError("Password is write-only")

    @password.setter
    def password(self, plaintext):
        salt = os.urandom(32)
        self._hashed_password = hashlib.pbkdf2_hmac( # Just an example. Does not need to be password related
            "sha256", plaintext.encode("utf-8"), salt, 100_000
        )

john = User("John", "secret")
john.password # AttributeError: attr. is write-only

Common `property()` use cases:
* **Validating Input Values:** Data validation is a common requirement in code that takes input from users or other sources that you could consider untrusted. Python’s `property()` provides a quick and reliable tool for dealing with input validation.


In [ ]:
class Point:
    def __init__(self, x):
        self.x = x

    @property
    def x(self):
        return self._x

    @x.setter
    def x(self, value):
        try:
            self._x = float(value)
            print("Validated!")
        except ValueError:
            raise ValueError('"x" must be a number') from None

* **Providing Computed Attributes:** If you need an attribute that builds its value dynamically whenever you access it, then using a property can be a great choice. These kinds of attributes are commonly known as computed attributes.

In [ ]:
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    @property
    def area(self): # Area is the computed attribute/property here
        return self.width * self.height

* **Caching Computed Attributes:** Sometimes you have a given computed attribute that you use frequently. To work around this problem, you can cache the computed value for later reuse.

In [ ]:
from time import sleep

class Circle:
    def __init__(self, radius):
        self.radius = radius

    @property
    def radius(self):
        return self._radius

    @radius.setter
    def radius(self, value):
        self._diameter = None
        self._radius = value

    @property
    def diameter(self): # Caching diameter property
        if self._diameter is None:
            sleep(0.5)  # Simulate a costly computation
            self._diameter = self._radius * 2
        return self._diameter

Another way to create cached properties is to use `functools.cached_property()` from the standard library.  This function works as a decorator and allows you to transform a method into a cached property. The property computes its value only once and caches it as a normal attribute during the lifetime of the instance:

In [ ]:
from functools import cached_property
from time import sleep

class Circle:
    def __init__(self, radius):
        self.radius = radius

    @cached_property
    def diameter(self):
        sleep(0.5)  # Simulate a costly computation
        return self.radius * 2

circle = Circle(3)
## Does not block attribute updates: (unless you provide a setter method)
circle.diameter
circle.diameter = 4

If you want to create a cached property that doesn’t allow modifications: `property()` + `functools.cache()`:

In [ ]:
from functools import cache
from time import sleep

class Circle:
    def __init__(self, radius):
        self.radius = radius

    @property
    @cache 
    def diameter(self): # Prevents changes
        sleep(0.5) # Simulate a costly computation
        return self.radius * 2

circle = Circle(3)

circle.diameter
circle.diameter = 4

* **Logging Attribute Access and Mutation:** (wip, maybe will finish when logging comes)

***

##### **- Overriding Properties in Subclasses** (wip)

***

##### [**Descriptors**](#managing-attributes-through-descriptors) 

Descriptors are Python objects that implement a method of the **descriptor protocol,** which gives you the ability to create objects that have special behavior when they are accessed as attributes of other objects. The descriptor protocol is defined as:

```python
__get__(self, obj, type=None) -> object
__set__(self, obj, value) -> None
__delete__(self, obj) -> None
__set_name__(self, owner, name)
```

If your descriptor implements just `.__get__()`, then it is said to be a **non-data** descriptor. If it implements `.__set__()` or `.__delete__()`, then it is said to be a **data** descriptor. This is not only a difference in name but in **behaviour.** A descriptor class needs at least the `.__get__()` method.

To define a descriptor protocol, for example:

In [ ]:
class ExampleAttribute: # This class implements the descriptor protocol.
    def __get__(self, obj, type=None):
        print("Accessing attribute to get its value...")
        return 1
    
    def __set__(self, obj, value):
        print("Accessing attribute to set its value...")
        raise AttributeError("Cannot set value") # As an example, we will only consider a read-only descriptor.

class Ex:
    attr = ExampleAttribute() # Descriptor class instantiated as class attribute of another.

myex = Ex()
myattr = myex.attr
print(myattr)


Once the class that implements the descriptor protocol is instantiated as a **class attribute** of another, it can be considered a **descriptor.** It has binding behavior when it is accessed using dot notation. 

**How Descriptors work in Python Internals**

- **Descriptors in Properties**

Properties in Python are just descriptors. If you want to get the same result without using descriptors, the best approach is to use **properties.** For example:

In [ ]:
class Ex:
    @property
    def attr(self):
        print("Accessing attribute to get its value...")
        return 5

    @attr.setter
    def attr(self, value):
        print("Accessing attribute to set its value...")
        raise AttributeError("Cannot set value")

myex = Ex()
myattr = myex.attr
print(myattr)

The example above makes use of decorators to define a property with attached **getter** and **setter** methods. It is equivalent to: 

In [ ]:
class Ex:
    def getter(self):
        print("Accessing the attribute to get its value")
        return 10

    def setter(self, value):
        print("Accessing the attribute to set its value")
        raise AttributeError("Cannot set value")

    attr = property(getter, setter)

myex = Ex()
myattr = myex.attr
print(myattr)

The property, as said before, was created using `property()`. It returns a **`property` object** that implements the descriptor protocol, using the `fget`, `fset`, and `fdel` for the implementation of the protocol's methods.

- **Descriptors in Methods and Functions**

We already know what methods are. Regular functions that have the first argument reserved for the object instance, `self`. The magic that transforms your `obj.method(*args)` call into `method(obj, *args)` is inside a `.__get__()` implementation of the function object that is, in fact, a **non-data descriptor.** The `function` object implements `.__get__()` so that it returns a bound method when you access it with dot notation. The (`*args`) that follow invoke the function. 

For example, from the official documentation:

In [ ]:
import types

class Function(object): # Not much to understand here, this is more advanced.
    ...
    def __get__(self, obj, objtype=None):
        "Simulate func_descr_get() in Objects/funcobject.c"
        if obj is None:
            return self
        return types.MethodType(self, obj)

When the function is accessed with dot notation, `.__get__()` is called and a bound method is returned. 

This works for regular instance methods just like it does for class methods or static methods.

(This section may need more info. in the future. Understanding descriptors in funcs/methods may be advanced OOP.)

- **Using Descriptors Properly** -   

    To use descriptors, the **protocol** of course must be implemented. The most important are `.__get__()` and `.__set__()`: 

    ```python
    __get__(self, obj, type=None) -> object
    __set__(self, obj, value) -> None
    ```

    - `self` is the instance of the descriptor you are writing.
    - `obj` is the instance of the object your descriptor is attached to
    - `type` is the type of the object the descriptor is attached to.

In `.__set__()`, you do not have the `type` variable, because you can only call `.__set__()` on the object. In contrast, you can call `.__get__()` on both the object and the class:

```python
Class.attr
my_class_obj.attr
```

Another important thing to know is that Python descriptors are **instantiated** just **once** per class. That means that every single instance of a class containing a descriptor shares that descriptor instance. For example:

In [ ]:
class OneDigit():
    def __init__(self):
        self.value = 0
    def __get__(self, obj, type=None):
        return self.value
    def __set__(self, obj, value):
        if value > 9 or value < 0 or int(value) != value:
            raise AttributeError("The value is invalid")
        self.value = value

class Foo(): # Each instance of Foo shares the same descriptor instance, number. Class-level attrib.
    number = OneDigit()

my_foo_object = Foo()
my_second_foo_object = Foo()

my_foo_object.number = 3
print(my_foo_object.number)
print(my_second_foo_object.number)

my_third_foo_object = Foo()
print(my_third_foo_object.number)

The best solution here is to simply not store values in the descriptor itself, but to store them in the **object** that the descriptor is attached to:

In [ ]:
class OneDigit():
    def __init__(self, name):
        self.name = name

    def __get__(self, obj, type=None):
        return obj.__dict__.get(self.name) or 0

    def __set__(self, obj, value):
        obj.__dict__[self.name] = value

class Foo():
    number = OneDigit("number")

my_foo_object = Foo()
my_second_foo_object = Foo()

my_foo_object.number = 3
print(my_foo_object.number)
print(my_second_foo_object.number)

my_third_foo_object = Foo()
my_third_foo_object.number = 10
print(my_third_foo_object.number)

In this example, when you set a value to the `number` attribute of your object, the descriptor stores it in the `.__dict__` attribute of the object it is attached to using the same name of the descriptor itself (`self.name`).

The only problem here is hat when you instantiate the descriptor you have to specify the name as a parameter (`"number"`). The descriptor protocol has a new method `.__set_name__()` that does all the magic for you:

```python
__set_name__(self, owner, name)
```

With this new method, whenever you instantiate a descriptor this method is called and the name parameter automatically set.

In [ ]:
class OneDigit():
    def __set_name__(self, owner, name): # init removed. set_name implemented
        self.name = name

    def __get__(self, obj, type=None) -> object:
        return obj.__dict__.get(self.name) or 0

    def __set__(self, obj, value) -> None:
        obj.__dict__[self.name] = value

class Foo():
    number = OneDigit()

my_foo_object = Foo()
my_second_foo_object = Foo()

my_foo_object.number = 3
print(my_foo_object.number)
print(my_second_foo_object.number)

my_third_foo_object = Foo()
print(my_third_foo_object.number)

This makes it possible to create your descriptor without specifying the name of the internal attribute that you use. For more info. about descriptors: [Special Methods\Descriptors](#managing-attributes-through-descriptors)

##### **Lightweight Classes with `.__slots__`** <a id="lightweight-classes-slots"></a>

In a Python class, using the `.__slots__` attribute can help you reduce the memory footprint of the corresponding instances. This attribute prevents the automatic creation of an instance `.__dict__.`

Using `.__slots__` is particularly handy when you have a class with a fixed set of attributes, and you’ll use that class to create a large number of objects.

In [ ]:
class Point:
    __slots__ = ("x", "y")
    def __init__(self, x, y):
        self.x = x
        self.y = y

point = Point(4, 8)
point.__dict__

#### **Providing Behaviour with Methods**

In a Python class, you can define three different types of methods:

1. **Instance methods**, which take the current instance, **self**, as their first argument
2. **Class methods**, which take the current class, **cls**, as their first argument
3. **Static methods**, which take neither the instance nor the class as an argument

##### **Class Methods with `@classmethod`**

A class method is a method that takes the **class object (`cls`)** as its first argument instead of taking `self`. You can create class methods using the `@classmethod` decorator. Providing your classes with **multiple constructors** is one of the most common use cases of class methods in Python.    

In [ ]:
class Point:
    def __init__(self, x, y, z):
        self.x = x
        self.y = y
        self.z = z

    #def __iter__(self):
        #yield from (self.x, self.y, self.z)

    @classmethod
    def from_sequence(cls, sequence):
        return cls(*sequence)

    #def __repr__(self):
        #return f"{type(self).__name__}({self.x}, {self.y}, {self.z})"

In [ ]:
Point.from_sequence((1, 2, 3))
point = Point(4, 5, 6)

To create the new object, you use the `cls` argument, which holds an implicit reference to the current class, which Python injects into your method automatically.

##### **Static Methods with `@staticmethod`**

A static method is a method that **does not** take an instance neither the class as an argument: they are regular functions defined within the class. To write a static method in a class, use the **`@staticmethod`** decorator:

In [ ]:
class Point:
    def __init__(self, x, y, z):
        self.x = x
        self.y = y
        self.z = z

    #def __iter__(self):
        #yield from (self.x, self.y, self.z)

    @classmethod
    def from_sequence(cls, sequence):
        return cls(*sequence)

    @staticmethod
    def show_intro_message(name):
        print(f"Hey {name}! This is your 3D Point!")

    #def __repr__(self):
        #return f"{type(self).__name__}({self.x}, {self.y}, {self.z})"

In [ ]:
point = Point(1, 2, 3)
point.show_intro_message("Eduardo")

**`@classmethod`** methods take `cls` (the class itself) instead of `self`. Their most common use is as **alternative constructors** — ways to build an instance from data in a different shape than `__init__` expects (e.g. `Patient.from_csv_row(row)`, or built-ins like `datetime.fromtimestamp(...)` and `dict.fromkeys(...)`). Using `cls(...)` instead of hardcoding the class name makes the constructor **polymorphic**: if a subclass calls the inherited classmethod, `cls` is the *subclass*, so the correct subtype gets built automatically.

**`@staticmethod`** methods take neither `self` nor `cls`. They're plain functions that happen to live in the class's namespace because they're conceptually related to it (e.g. a small validation helper). Use one when a piece of logic belongs next to the class organizationally, but never needs to touch instance or class state.

In [ ]:
class Patient:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def __repr__(self):
        return f"{type(self).__name__}({self.name!r}, {self.age})"

    @classmethod
    def from_csv_row(cls, row):
        # Alternative constructor: builds an instance from "name,age" instead of separate args.
        # cls(...) instead of Patient(...) is what makes this polymorphic (see VIPPatient below).
        name, age = row.split(",")
        return cls(name, int(age))

    @staticmethod
    def is_valid_age(age):
        # Pure utility: doesn't need self or cls, just grouped here for organization
        return 0 <= age <= 130

print(Patient.from_csv_row("Ana,34"))
print(Patient.is_valid_age(34), Patient.is_valid_age(200))

class VIPPatient(Patient):
    pass

# VIPPatient inherits from_csv_row unchanged, but cls is now VIPPatient:
print(VIPPatient.from_csv_row("Luis,50"))  # Builds a VIPPatient, not a plain Patient

##### **Debugging Classes**

You’ll find a few **exceptions** that can occur when working with Python classes. These are some of the most common ones:

* **`AttributeError`:** occurs when the object does not define the attribute or method you are trying to access.
* **`TypeError`:** occurs when you apply a function to an object that does not support it.
* **`NotImplementedError`:** occurs when an abstract method isn’t implemented in a concrete subclass.

### 4.3 Dunder/Magic (Special) Methods — The Data Model

In Python, special methods are also called **magic** methods, or **dunder** methods. "Dunder" refers to a naming convention used to name this special methods and attributes: **double leading and trailing** underscores.

Python automatically calls magic methods as a response to certain operations. Magic methods support core object-oriented features in Python.

Official documentation definition: 

> A method that is called implicitly by Python to execute a certain operation on a type, such as addition. Such methods have names starting and ending with double underscores.

"Dunder" (double-underscore) methods are how Python objects plug into the language's own syntax. This is often called Python's **data model:** when you write `len(obj)`, Python calls `obj.__len__()`; `a + b` calls `a.__add__(b)`; `str(obj)` calls `obj.__str__()`. This is exactly how a built-in `list` and your own custom class can *both* respond to `len()` — both implement `__len__`. You'll almost never call these methods directly; you use the ordinary syntax/built-in and Python calls them for you behind the scenes. Even though one can call them directly, special methods are **not** intended for **direct** use.

#### **Controlling the Object Creation Process**

##### **`__init__` (Object Initializer)**

Python calls the `.__init__()` method whenever you call the constructor of a given class. This method works as an **initializer** because it allows you to provide **initial** values (initialize) to any instance attributes that you define in your classes.

```python
class MyClass(...):
    def __init__(self, attr)
        self.attr = attr
```

##### **`__new__` (Object Creator)**

Python implicitly calls the `.__new__()` method whenever you call a class constructor as a first step in the **instantiation** process. This method is responsible for **creating** and **returning** a new **empty** object of the underlying class. Python then passes this new object to `.__init__()` for initialization.

Most useful in more advanced cases, i.e. creating subclasses of immutable types such as `int`, `float`, `tuple` and `str`.

#### **Representing Objects as Strings: `__repr__` vs `__str__`**

In some cases, your program may need to display useful info. to the user, while in other, you may need to show info. to other programmers using your code.  If you want to provide user-friendly output, then you can use the `.__str__()` method. On the other hand, when you need to provide developer-friendly output, then you can use the `.__repr__()` method

##### **`.__str__()`**

The `.__str__()` special method returns a human-readable string representation of the object at hand. Python calls this method when you call the built-in `str()` function. 

Python also calls this method when you pass the instance as an argument to `print()` or `format()`.

In [ ]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def __str__(self):
        return f"I'm {self.name}, and I'm {self.age} years old."

bob = Person("Bob", 20)
print(bob)
str(bob)

##### **`.__repr__()`**

The `.__repr__()` method returns a string representation of an object that’s targeted at the developer. The string returned should be crafted in such a way that you can construct an identical instance of the object from the string.

In [ ]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def __repr__(self):
        return f"{type(self).__name__}(name='{self.name}', age={self.age})"

bob = Person("Bob", 20)
repr(bob) # From this representation, one can build an identical instance of the object.

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __repr__(self):
        return f"Point(x={self.x}, y={self.y})"   # unambiguous, dev-facing

    def __str__(self):
        return f"({self.x}, {self.y})"             # friendly, user-facing

p = Point(1, 2)
print(p)          # print() uses __str__: (1, 2)
print(repr(p))    # repr() uses __repr__: Point(x=1, y=2)
print([p, p])     # Containers always show repr() of their elements

* **`__repr__`** should be unambiguous — ideally a valid Python expression that could recreate the object. It's used by the REPL, by `repr()`, by debuggers, and to display objects *inside* containers (printing a list shows each element's `repr`, not its `str`). Every class should define this.
* **`__str__`** should be readable/user-facing. It's used by `str()`, `print()`, and f-strings. If `__str__` is missing, Python falls back to `__repr__` — so defining just `__repr__` already covers both cases reasonably; add a separate `__str__` only when you want a friendlier presentation for end users specifically.

#### **Operator Overloading**

Python has multiple types of **operators,** which are special symbols, combinations of symbols, or keywords that designate some type of computation. Internally, **Python supports operators with special methods.**

Operator **overloading** means providing additional **functionality** to these operators.

##### **Arithmetic Operators**

Used to perform arithmetic operations on **numeric** values.

| Operator | Supporting  Special Method |
|----------|-------------------|
| `+`      | `.__add__(self, other)` |
| `-`      | `.__sub__(self, other)` |
| `*`      | `.__mul__(self, other)` |
| `/`      | `.__truediv__(self, other)` |
| `//`     | `.__floordiv__(self, other)` |
| `%`      | `.__mod__(self, other)` |
| `**`     | `.__pow__(self, other[, modulo])` |

Note that all these methods take a second argument called **`other`**. In most cases, this argument should be the same **type** as **`self`** or a compatible type.

The special methods that support arithmetic operators are affected by the relative **position** of each object in the expression. Python calls these methods on the **left-hand** operand by default. If this operand does not implement the method, or does it but does not behave correctly, Python will fall to the **right-hand** version of the operator method: **`.__r*__()`.**

In [ ]:
class Number: # I want to build a class that supports additions between my objects and int/float
    def __init__(self, value):
        self.value = value

    def __add__(self, other): # Will get invoked when using a instance of Number as left-hand op.
        print("__add__ called")
        if isinstance(other, Number):
            return Number(self.value + other.value)
        elif isinstance(other, int | float):
            return Number(self.value + other)
        else:
            raise TypeError("unsupported operand type for +")

    def __radd__(self, other): # Will get invoked when using a instance of Number as right-hand op.
        print("__radd__ called")
        return self.__add__(other)

In [ ]:
one = Number(1)
three = Number(3)

one + three # __add__ either way
print('-' * 10)
one + 1 # __add__ 
print('-' * 10)
2 + three #__radd__ -> __add__

So, Python calls these methods when the left-hand operand does not support the corresponding operation and the **operands** are of **different types.**

Python also has some **unary operators.** It calls them unary because they operate on a **single** operand:

| Operator | Supporting  Special Method |
|----------|-------------------|
| `-`      | `.__neg__(self)` |
| `+`      | `.__pos__(self)` |

Arithmetic and other operators are each backed by a dunder: `+` → `__add__`, `-` → `__sub__`, `*` → `__mul__`, and so on. If the left operand's method returns the special singleton `NotImplemented` (not an exception — a specific return value), Python then tries the **reflected** method on the right operand (`__radd__`, `__rmul__`, ...). This is exactly what lets `my_obj + 5` *and* `5 + my_obj` both work, even though `int` knows nothing about your class: `int.__add__` gives up with `NotImplemented`, and Python then tries `my_obj.__radd__(5)`. In-place variants (`__iadd__` for `+=`) also exist but are optional — without one, Python just falls back to `__add__` followed by reassignment.

In [ ]:
class Vector:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

    def __add__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented   # let Python try other.__radd__(self), or raise TypeError
        return Vector(self.x + other.x, self.y + other.y)

    def __mul__(self, scalar):
        return Vector(self.x * scalar, self.y * scalar)   # Vector * number

    def __rmul__(self, scalar):
        # number * Vector: int.__mul__ gives up (NotImplemented), Python retries here
        return self.__mul__(scalar)

v1, v2 = Vector(1, 2), Vector(3, 4)
print(v1 + v2)     # __add__
print(v1 * 3)        # __mul__
print(3 * v1)         # __rmul__

try:
    v1 + "not a vector"
except TypeError as e:
    print("Rejected:", e)   # __add__ -> NotImplemented, str has no matching __radd__ either -> TypeError

##### **Comparison Operator Methods**

Special method also support **comparison operators.** All the operators and its support methods are shown below: 

| Operator | Supporting  Special Method |
|----------|-------------------|
| `<`      | `.__lt__(self, other)` |
| `<=`      | `.__le__(self, other)` |
| `==`      | `.__eq__(self, other)` |
| `!=`      | `.__ne__(self, other)` |
| `>=`     | `.__ge__(self, other)` |
| `>`      | `.__gt__(self, other)` |

In [ ]:
class Rectangle:
    def __init__(self, height, width):
        self.height = height
        self.width = width

    def area(self): 
        return self.height * self.width

    def __eq__(self, other):
        return self.area() == other.area()

    def __lt__(self, other):
        return self.area() < other.area()

    def __gt__(self, other):
        return self.area() > other.area()

In [ ]:
r = Rectangle(1, 3)
s = Rectangle(2, 5)
print(r < s)
print(r == s)

By default, `==` on custom objects compares **identity** (same as `is`) — two separate instances with identical data are *not* equal until you override `__eq__` to compare values instead. Each comparison operator has its own dunder (`__lt__`, `__le__`, `__gt__`, `__ge__`); none is automatically derived from another. `functools.total_ordering` is a class decorator that fills in the rest for you, as long as you provide `__eq__` and just *one* of the four — much less boilerplate.

**`__hash__`** is what makes an object usable as a dict key or set element. 

**Important rule:** if you override `__eq__`, Python automatically sets `__hash__` to `None` (making the object unhashable) — because the default identity-based hash would now be inconsistent with your new, value-based equality. If you want equal objects to also be hashable, you must define `__hash__` explicitly yourself, deriving it from the *same* fields used in `__eq__` (equal objects must always produce the same hash). As a rule of thumb, only make **immutable** objects hashable — if a hashed object's fields change afterward, it can effectively "get lost" inside the set/dict that hashed it under its old value.

In [ ]:
from functools import total_ordering

@total_ordering  # Fills in the missing comparison/ordering methods
class Money:
    def __init__(self, amount):
        self.amount = amount

    def __repr__(self):
        return f"Money({self.amount})"

    def __eq__(self, other):
        return self.amount == other.amount

    def __lt__(self, other):
        return self.amount < other.amount

    def __hash__(self):
        return hash(self.amount)  # Consistent with __eq__: equal Moneys hash the same

prices = [Money(30), Money(10), Money(20)]
print(sorted(prices))           # total_ordering derives __le__/__gt__/__ge__ from __eq__ + __lt__
print(Money(10) == Money(10))   # True: value equality, not identity
print(Money(10) < Money(30) <= Money(30))
print({Money(10), Money(10), Money(20)})   # Set: duplicates collapse (equal AND same hash)

##### **Membership Operators**

Python has two operators that allow you to determine whether a given value is in a collection of values. The operators are **`in`** and **`not in`.** They support a check that’s known as a 

Both of these operators are supported by the special method **`__contains__`.** By implementing this method in a custom class, you can control how its instances will behave in a membership test.

In [ ]:
class Stack:
    def __init__(self, items=None):
        self.items = list(items) if items is not None else []

    def push(self, item):
        self.items.append(item)

    def pop(self):
        return self.items.pop()

    def __contains__(self, item):
        for current_item in self.items:
            if item == current_item:
                return True
        return False

stack = Stack([1, 2, 3, 4])
print(1 in stack)
print(2 not in stack)

##### **Augmented Assignments**

Common expressions that use the **current** value variable to update the variable **itself:**

```python
counter = 0
counter = counter + 1
...
```
These types of operations have their shortcuts and have corresponding supporting special methods:

| Operator | Supporting  Special Method |
|----------|-------------------|
| `+=`      | `.__iadd__(self, other)` |
| `-=`      | `.__isub__(self, other)` |
| `*=`      | `.__imul__(self, other)` |
| ```...```     | ```...``` |

In [ ]:
class Stack:
    def __init__(self, items=None):
            self.items = list(items) if items is not None else []

    def __add__(self, other):
        return type(self)(self.items + other.items)

    def __iadd__(self, other):
        self.items.extend(other.items)
        return self

    def __repr__(self):
        return f"{type(self).__name__}({self.items!r})"

stack = Stack([1, 2, 3])
stack += Stack([4, 5, 6])
print(stack)

#### **Introspecting your Objects**

Some special methods also support **instrospection** in your custom classes. These dunders are useful for this purpose:

| Method | Description |
|----------|-------------------|
| `.__dir__()`      | Returns a list of attributes and methods of an object |
| `.__hasattr__()`      | Checks whether an object has a specific attribute |
| `.__instancecheck__()`      | Checks whether an object is an instance of a certain class |
| `.__subclasscheck__()`     | Checks whether a class is a subclass of a certain class |

With these methods one can provide custom behaviours and control the introspection process.
For example:

In [ ]:
class Rectangle:
    # ...
    def __init__(self, height, width):
        self.height = height
        self.width = width

    def __dir__(self):
        print("__dir__ called")
        return sorted(self.__dict__.keys())

t = Rectangle(1, 2)
print(dir(t))

#### **Controlling Attribute Access**

Behind every attribute access, assignment, and deletion, there are special methods that support the specific operation. The involved methods are:

| Method | Description |
|----------|-------------------|
| `.__getattribute__(self, name)` | Runs when you access an attribute called `name` |
| `.__getattr__(self, name)`| Runs when you access an attribute that doesn’t exist in the current object (runs as a fallback) |
| `.__setattr__(self, name, value)` | Runs when you assign `value` to the attribute called `name` |
| `.__delattr__(self, name)` | Runs when you delete the attribute called `name` |

You could use these methods to inspect and customize how you access, set, and delete attributes in your custom classes.

##### **Retrieving Attributes**

Python has two methods to handle attribute access: `.__getattribute__()` method runs on **every** attribute access, while the `.__getattr__()` method only runs if the target attribute **does not exist** in the current object.

For example:

In [ ]:
import math

class Circle:
    def __init__(self, radius):
        self.radius = radius

    def __getattribute__(self, name):
        print(f"__getattribute__ called for {name}")
        return super().__getattribute__(name) # Delegate the attr. access to the parent class

    def __getattr__(self, name):
        print(f"__getattr__ called for {name}")
        if name == "diameter":
            return self.radius * 2
        return super().__getattr__(name) # Delegate the attr. access to the parent class

circle = Circle(1)

In [ ]:
circle.radius
circle.diameter

##### **Setting Attributes**

Setting an attribute to a given value is done using the **assignment** operator (=). Python detects this assignment and internally calls the `.__setattr()__` special method.

Following the example of above:

In [ ]:
import math

class Circle:
    def __init__(self, radius):
        self.radius = radius # __init__ also runs the attr. assignment

    # ...

    def __setattr__(self, name, value):
        if name == "radius":
            if not isinstance(value, int | float):
                raise TypeError("radius must be a number")
            if value <= 0:
                raise ValueError("radius must be positive")
        super().__setattr__(name, value)

circle = Circle(5)

In [ ]:
circle.radius = 10
circle.radius = "10" 

##### **Deleting Attributes**

Sometimes, you need to delete a given attribute from an object after you use it. To control how a class responds to attribute **deletion** with the `del`statement, one can use the `.__delattr__()` special method.

For example:

In [ ]:
class NonDeletable:
    def __init__(self, value):
        self.value = value

    def __delattr__(self, name):
        raise AttributeError(
            f"{type(self).__name__} doesn't support attribute deletion")

one = NonDeletable(1)

In [ ]:
print(one.value)
del one.value

#### **Managing Attributes through Descriptors**

Python [**descriptors**](#Descriptors) are classes that allow you to have fine control over the attributes of a given custom class. You can use descriptors to add **function-like behavior** on top of the attributes of a given class.

A descriptor class needs at least the **`.__get__()`** special method. However, the full descriptor protocol consists of the following methods:

| Method | Description |
|----------|-------------------|
| `.__get__(self, instance, type=None)` | **Getter** method that allows you to retrieve the current value of the managed attribute |
| `.__set__(self, instance, value)`| **Setter** method that allows you to set a new value to the managed attribute |
| `.__delete__(self, instance)` | **Deleter** method that allows you to remove the managed attribute from the containing class |
| `.__set_name__(self, owner, name)` | **Name setter** method that allows you to define a name for the managed attribute |

Say you have as an example a couple of classes representing shapes: Circle, Square, etc. and you need to validate some parameters. One can use a **descriptor** to perform the **validation logic:**

In [ ]:
class PositiveNumber:
    def __set_name__(self, owner, name):
        self._name = name

    def __get__(self, instance, owner):
        return instance.__dict__[self._name]

    def __set__(self, instance, value):
        if not isinstance(value, int | float) or value <= 0:
            raise ValueError("positive number expected")
        instance.__dict__[self._name] = value

class Circle:
    radius = PositiveNumber()

    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return round(math.pi * self.radius**2, 2)

class Square:
    side = PositiveNumber()

    def __init__(self, side):
        self.side = side

    def calculate_area(self):
        return self.side**2

circle1 = Circle(1)
square1 = Square(2)
circle2 = circle(-2)

Note that the managed attributes must be **class attributes** with an instance attribute counterpart. Compare this example to a solution using properties:

In [ ]:
class Circle:
    def __init__(self, radius):
        self.radius = radius

    @property
    def radius(self):
        return self._radius

    @radius.setter
    def radius(self, value):
        if not isinstance(value, int | float) or value <= 0:
            raise ValueError("positive number expected")
        self._radius = value

    def calculate_area(self):
        return 3.14 * self._radius**2

class Square:
    def __init__(self, side):
        self.side = side

    @property
    def side(self):
        return self._side

    @side.setter
    def side(self, value):
        if not isinstance(value, int | float) or value <= 0:
            raise ValueError("positive number expected")
        self._side = value

    def calculate_area(self):
        return self._side**2

Maybe, to avoid code repetition, **descriptors** can turn out to be a good idea.

#### **Implementing Custom Sequences and Mappings**

Sequences (lists, tuples, strings...) and mappings (dicts) are fundamental data types. You can create your own sequence-like and mapping-like classes by implementing the required special methods. 

For this,  Python defines the sequence and mapping protocols. These protocols are collections of special methods. For example, for the sequence protocol:

| Method | Description |
|----------|-------------------|
| `.__getitem__()` | Called when you access an item using indexing like in `sequence[index]` |
| `.__len__()`| Called when you invoke the built-in `len()` function to get the number of items in the underlying sequence |
| `.__contains__()	` | Called when you use the sequence in a membership test with the `in` or `not in` operator |
| `.__reversed__()` | Called when you use the built-in `reversed()` function to get a reversed version of the underlying sequence |

* **`__len__`** backs `len(obj)`; it must return a non-negative `int`.
* **`__getitem__`** backs indexing (`obj[i]`). If `__iter__` is *not* defined, it also transparently backs iteration (`for x in obj`) and even `in`: Python falls back to calling `__getitem__(0)`, `__getitem__(1)`, ... until it hits `IndexError`.
* **`__contains__`** backs the `in` operator directly. Without it, Python falls back to iterating (or to the `__getitem__` fallback above) and comparing each element for equality. Defining `__contains__` explicitly lets you make membership tests faster (e.g. an O(1) lookup) instead of relying on the O(n) fallback.

In [ ]:
class Playlist:
    def __init__(self, songs):
        self._songs = list(songs)

    def __len__(self):
        return len(self._songs)

    def __getitem__(self, index):
        return self._songs[index]

    def __contains__(self, song):
        print("Custom __contains__ check")
        return song in self._songs

    def __reversed__(self):
        return type(self)(reversed(self._songs))

pl = Playlist(["Song A", "Song B", "Song C"])
print(len(pl))              # __len__
print(pl[1])                  # __getitem__
print(pl[0:2])                 # slicing works too, forwarded to the underlying list
print("Song B" in pl)          # __contains__: prints the debug line

for song in pl:                # no __iter__ defined: falls back to repeated __getitem__ calls
    print(song)

#### **Making your Objects Callable: `__call__`**

A **callable** is any object that you can call using a pair of **parentheses** and optionally a seiries of arguments. For example: functions, classes and methods.

To create your own callable, you can add the `__call__` special method to a class. Any **instance** of a class with a `.__call__()` method behaves like a **function.** 

Defining `__call__` makes instances **callable**, just like functions: `obj(...)` invokes `obj.__call__(...)`. This is useful for objects that "are" a function but need to carry their own configuration/state along with them — an alternative to closures, or to a class whose entire purpose is a single "do the thing" method (e.g. configurable transformations, memoizing wrappers, class-based decorators).

In [ ]:
class Multiplier:
    def __init__(self, factor):
        self.factor = factor  # configuration carried by the instance

    def __call__(self, value):
        return value * self.factor

double = Multiplier(2)
triple = Multiplier(3)

print(double(5))         # calls double.__call__(5) -> 10
print(triple(5))         # 15
print(callable(double))  # True: has __call__

Two more major dunder pairs — **`__iter__`/`__next__`** (the iterator protocol) and **`__enter__`/`__exit__`** (the `with` statement / context manager protocol) — are deliberately left for the next block on **Iterators, Generators & Context Managers**, where they make a lot more sense alongside `yield` and resource management.

### 4.4 Inheritance

Inheritance is a powerful feature of object-oriented programming. It consists of creating **hierarchical** relationships between classes, where child classes inherit attributes and methods from their parent class.

Implementing inheritance lets you reduce duplication and repetition by code reuse.

#### **Simple Inheritance**

**Inheritance** lets you define a new class (the **subclass**/child) based on an existing one (the **superclass**/parent), automatically getting all of its attributes and methods, so you can add new ones or override existing ones instead of writing everything from scratch.

When you have a class that inherits from a single parent class, then you are using **single-base**/simple inheritance. The syntax to define these is:

```python
class Parent:
    # Parent definition goes here...
    <body>

class Child(Parent):
    # Child definitions go here...
    <body>
```

Parent classes typically provide **generic** and **common** functionality that you can reuse throughout multiple child classes. `Child` is the class that inherits attributes and methods from `Parent`. **Superclass** or **base class** are equivalent to parent class; **derived class** or **subclass** are to child class.

In [ ]:
class Vehicle: # Provides the common interface of all your subclasses
    def __init__(self, make, model, year): # Will provide these common attributes
        self.make = make
        self.model = model
        self.year = year
        self._started = False

    def start(self): # Common method provided
        print("Starting engine...")
        self._started = True

    def stop(self): # Common method provided
        print("Stopping engine...")
        self._started = False

class Car(Vehicle): # Child class from Vehicle
    def __init__(self, make, model, year, num_seats):
        super().__init__(make, model, year)
        self.num_seats = num_seats

    def drive(self):
        print(f'Driving my "{self.make} - {self.model}" on the road')

    def __str__(self):
        return f'"{self.make} - {self.model}" has {self.num_seats} seats'

tesla = Car("Tesla", "Model S", 2022, 5)
tesla.start()
tesla.drive()
print(tesla)

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        raise NotImplementedError("Subclasses must implement speak()")

    def describe(self):
        return f"{self.name} is an animal."

class Cat(Animal):
    def __init__(self, name, indoor=True):
        super().__init__(name)     # runs Animal.__init__, avoids repeating "self.name = name"
        self.indoor = indoor

    def speak(self):                # override: Cat's own version replaces Animal's
        return f"{self.name} says Meow!"

    def describe(self):
        base = super().describe()   # call the parent's version, then extend it
        return base + " Specifically, a cat."

whiskers = Cat("Whiskers")
print(whiskers.speak())
print(whiskers.describe())
print(isinstance(whiskers, Animal), isinstance(whiskers, Cat))
print(type(whiskers) is Animal)     # False: type() ignores ancestors, isinstance() doesn't

**`super()`** returns a proxy that lets you call the parent's version of a method — most commonly to *extend* `__init__` (run the parent's setup, then add more) rather than duplicating its logic. The `super()` function lets you access members of the superclass as it name suggests.
A subclass can freely **override** a method by redefining it with the same name; its version then takes precedence when called on a subclass instance, while `super()` still lets you reach the parent's original implementation from inside the override.

`isinstance(obj, Cls)` checks the *entire* inheritance chain (a `Cat` instance IS an `Animal` too), whereas `type(obj) is Cls` only matches the exact, most specific class.

##### **Class Hierarchies**

With inheritance, one can design class hierarchies, also known as **inheritance trees.** A class hierarchy is a set of closely **related classes** that are connected through inheritance and arranged in a **tree-like** structure.

The class or classes at the top of the hierarchy are the **base** classes, while the classes below are **derived** classes or subclasses. Inheritance-based hierarchies express an **is-a-type-of** relationship between subclasses and their base classes.

Each level in the hierarchy will inherit attributes and behaviors from the above levels. Classes at the top will be more generic with common functionality, while classes down will be more specialized.

In [ ]:
class Animal: # Base class of the hierarchy
    def __init__(self, name, sex, habitat):
        self.name = name
        self.sex = sex
        self.habitat = habitat

class Mammal(Animal):
    unique_feature = "Mammary glands"

class Bird(Animal):
    unique_feature = "Feathers"

class Fish(Animal):
    unique_feature = "Gills"

class Dog(Mammal):
    def walk(self):
        print("The dog is walking")

class Eagle(Bird):
    def fly(self):
        print("The eagle is flying")

class Salmon(Fish):
    def swim(self):
        print("The salmon is swimming")


Each level in the hierarchy can—and typically will—add new attributes and functionality on top of those that its parents already provide.

##### **Extended, Overridden Methods**

In some situations, a parent class may provide a given functionality only at a basic level, and you may want to **extend** that functionality in your subclasses. In other situations, the feature in the parent class is **not appropriate** for the subclass.

In these situations, you can use one of the following strategies:

* **Extending** an inherited method in a subclass: reuse some functionality provided by the superclass and add new functionality on top.
* **Overridding** an inherited method in a subclass: discard some functionality from the superclass and provide a new one in the subclass.

An example applying the first strategy goes as follows:

In [ ]:
class Aircraft:
    def __init__(self, thrust, lift, max_speed):
        self.thrust = thrust
        self.lift = lift
        self.max_speed = max_speed

    def show_technical_specs(self):
        print(f"Thrust: {self.thrust} kW")
        print(f"Lift: {self.lift} kg")
        print(f"Max speed: {self.max_speed} km/h")

class Helicopter(Aircraft):
    def __init__(self, thrust, lift, max_speed, num_rotors):
        super().__init__(thrust, lift, max_speed) # Extending the __init__ method from Aircraft
        self.num_rotors = num_rotors

    def show_technical_specs(self):
        super().show_technical_specs() # Extending this method from Aircraft
        print(f"Number of rotors: {self.num_rotors}")

helicopter = ...
helicopter.show_technical_specs()

For method overriding:

In [ ]:
class Worker:
    def __init__(self, name, address, hourly_salary):
        self.name = name
        self.address = address
        self.hourly_salary = hourly_salary

    def show_profile(self):
        print("== Worker profile ==")
        print(f"Name: {self.name}")
        print(f"Address: {self.address}")
        print(f"Hourly salary: {self.hourly_salary}")

    def calculate_payroll(self, hours=40):
        return self.hourly_salary * hours

class Manager(Worker):
    def __init__(self, name, address, hourly_salary, hourly_bonus):
        super().__init__(name, address, hourly_salary)
        self.hourly_bonus = hourly_bonus

    def calculate_payroll(self, hours=40): # Method overriding: different implementation
        return (self.hourly_salary + self.hourly_bonus) * hours

#### **Multiple Inheritance**

This type of inheritance allows you to create a class that inherits from **several** parent classes. The subclass will inherit attributes and methods from **all** its parents.

Allows you to reuse code from other existing classes. However, the complexity must be managed to avoid problems like the [diamond problem](https://en.wikipedia.org/wiki/Multiple_inheritance#The_diamond_problem).

In [ ]:
class Vehicle:
    def __init__(self, make, model, color):
        self.make = make
        self.model = model
        self.color = color

    def start(self):
        print("Starting the engine...")

    def stop(self):
        print("Stopping the engine...")

    def show_technical_specs(self):
        print(f"Make: {self.make}")
        print(f"Model: {self.model}")
        print(f"Color: {self.color}")

class Car(Vehicle):
    def drive(self):
        print("Driving on the road...")

class Aircraft(Vehicle):
    def fly(self):
        print("Flying in the sky...")

class FlyingCar(Car, Aircraft): # Inherits from both Car and Aircraft, which both inherit from Vehicle
    pass

flycar = FlyingCar("Space", "Flyer", "Red")
flycar.start()
flycar.drive()
flycar.fly()

##### **Method Resolution Order (MRO)**

When using multiple inheritance, a situation can happen where one class inherits from two or more classes that have the same base class: D -> (B & C) -> A. This is the **"diamond problem".**
The real issue appears when multiple parents provide specific versions of the same method.

To deal with this, Python uses MRO. It is an **algorithm** that Python follows to search for inherited methods in a multiple inheritance context. The **MRO (Method Resolution Order)** is the order Python searches through a class and its ancestors when looking up an attribute or method. MRO determines **which implementation** of a method or attribute to use when there are multiple versions of it in a class hierarchy.

In general, Python searches for methods and attributes in the following **order:**

1. The current class
2. The leftmost superclasses
3. The superclass listed next, from left to right, up to the last superclass
4. The superclasses of inherited classes
5. The `object` class

For plain single inheritance, it's simply child → parent → grandparent → ... → `object`. With **multiple inheritance**, Python computes this order using the **C3 linearization** algorithm, which guarantees children always come before parents and that the relative order of the listed parents is preserved. You don't need to memorize C3 — `ClassName.__mro__` always shows you the exact, predictable order Python will search.

In [ ]:
class A: 
    def method(self):
        print("A.method()")

class B(A): # Inherits from A
    def method(self):
        print("B.method()")
                        # Different versions of method() defined for every superclass (A, B, C)
class C(A): # Inherits from A
    def method(self):
        print("C.method()")

class D(B, C): # D inherits from B and C
    pass

D().method() # Python found method() first on B. This is the version ending up being called

The **`.__mro__`** special attribute lets you check the **current** MRO of a given class:

In [ ]:
print(D.__mro__) # Check the current MRO for a given class

##### **Mixin Classes**

A **mixin** class is a class that provides methods that you can reuse in other classes. They do not define new types: they are only meant to be inherited, **not** even **instantiatied.** The purpose is to use their functionality to add extra features to other classes. You can access the functionality of a mixin class in different ways. One of these ways is inheritance.

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        raise NotImplementedError("Subclasses must implement speak()")

    def describe(self):
        return f"{self.name} is an animal."

class SerializableMixin:
    def to_dict(self):
        # Works on ANY class that mixes this in, as long as it has instance attributes
        return dict(self.__dict__)

class Loggable:
    def log(self):
        print(f"[LOG] {self!r}")

class LoggedCat(Animal, SerializableMixin, Loggable):
    def speak(self):
        return f"{self.name} says Meow!"

nina = LoggedCat("Nina")   # __init__ is found via MRO on Animal, the first ancestor that defines it
print(nina.speak())
print(nina.to_dict())        # Comes from SerializableMixin
nina.log()                    # Comes from Loggable

print(LoggedCat.__mro__)      # The exact lookup order Python follows

So, it is a small class not meant to be used standalone, but **"mixed in"** via multiple **inheritance** to add one specific, reusable piece of behavior to other classes (by convention, its name often ends in **`Mixin`**). The roadmap advice holds: do not over-engineer deep multiple-inheritance hierarchies — just know how to read `__mro__` when you need to.

**Important:** as a result of how **Mixin** classes are defined, they do **not** imply an **is-a** type of relationship because they do not define a type of object.

For more information about Mixins (more advanced, and more examples, may not be needed): [Mixins in Python.](https://realpython.com/python-mixin/)

#### **Alternatives to Inheritance**

Inheritance is not the only technique that allows you to reuse functionality in object-oriented programming. There is also composition and delegation, which represent otehr types of relationships between classes.

##### **Composition**

Composition is used to represent or model a **has-a** relationship between classes. Through composition, you can create complex objects by combining objects that will work as components. These components may not make sense as standalone objects.

Favoring composition over inheritance leads to more flexible class designs. Composition is defined at **runtime,** which means that you can **dynamically replace** a current component with another component of the same type.

For example: 

In [ ]:
class IndustrialRobot: # Class built from its components: class Body, class Arm
    def __init__(self):
        self.body = Body() # Component
        self.arm = Arm() # Component

    def rotate_body_left(self, degrees=10):
        self.body.rotate_left(degrees)

    def rotate_body_right(self, degrees=10):
        self.body.rotate_right(degrees)

    def move_arm_up(self, distance=10):
        self.arm.move_up(distance)

    def move_arm_down(self, distance=10):
        self.arm.move_down(distance)

    def weld(self):
        self.arm.weld()

class Body:
    def __init__(self):
        self.rotation = 0

    def rotate_left(self, degrees=10):
        self.rotation -= degrees
        print(f"Rotating body {degrees} degrees to the left...")

    def rotate_right(self, degrees=10):
        self.rotation += degrees
        print(f"Rotating body {degrees} degrees to the right...")

class Arm:
    def __init__(self):
        self.position = 0

    def move_up(self, distance=1):
        self.position += distance
        print(f"Moving arm {distance} cm up...")

    def move_down(self, distance=1):
        self.position -= distance
        print(f"Moving arm {distance} cm down...")

    def weld(self):
        print("Welding...")

robot = IndustrialRobot()
print(robot.rotate_body_left())
print(robot.rotate_body_right(20))
print(robot.move_arm_down(5))
robot.weld()
robot.rotate_left() # Not like inheritance. Cannot access other classes methods directly.

Unlike inheritance, composition does not expose the entire interface of components, so it preserves **encapsulation.** They access and use only the required functionality from its components, resulting in more **robust** and reliable class design.

##### **Delegation**

With delegation, one can model a **can-do** relationship between classes, where an object hands a task to another, which does the task execution. Note that the delegated object can exist independently from the delegator.

You can use delegation to reuse code, separate concerns and achieve modularity. For example:

In [ ]:
class Stack: # Will delegate all its operations to the list object, which already knows
    def __init__(self, items=None):
        if items is None:
            self._items = []
        else:
            self._items = list(items) # Delegate all operations to the list() object: store, add, delete

    def push(self, item):
        self._items.append(item) # Operation delegated to this line

    def pop(self):
        return self._items.pop() # Operation delegated to this line

    def __repr__(self):
        return f"{type(self).__name__}({self._items})"


stack = Stack([1, 2, 3])
stack.push(4)
stack.pop()
stack

If you use delegation to write a class, the internal implementation of the delegated object will not be visible or directly accessible in the delegator, which preserves encapsulation.
If inheritance was used instead:

In [ ]:
class Stack(list): # Stack will inherit all functionality from list.
    def push(self, item):
        self.append(item)

    def pop(self):
        return super().pop()
    
    def __repr__(self) -> str:
        return f"{type(self).__name__}({super().__repr__()})"

stack = Stack()
dir(stack) # Exposed as part of the public class

With inheritance, the internals of parent classes are visible to subclasses, which breaks encapsulation. In these situations, composition or delegation are beter options.

Lastly, you can also impplement delegation with one of two special methods. `.__getattribute__()` is called whenever an instance or method is accessed. If the requested is not found, `.__getattr__()` is called. You can use this method to redirect the request to another object that can provide the attribute or method:

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        raise NotImplementedError("Subclasses must implement speak()")

    def describe(self):
        return f"{self.name} is an animal."

class Serializable:
    def __init__(self, instance):
        self.instance = instance

    def to_dict(self):
        # Works on ANY class that mixes this in, as long as it has instance attributes.
        return dict(self.__dict__)

# ...

class Cat(Animal, Serializable): # See the difference here between inheriting only Animal and both.
    def __init__(self, name, age):
        super().__init__(name)
        self.age = age

    def speak(self):
        return f"{self.name} says Meow!"

    def __getattr__(self, name): # Delegated to other object: Serializer. 
        return getattr(Serializable(self), name)

cat = Cat("Joe", 5)
print(cat.speak())
print(cat.to_dict())

##### **Dependency Injection**

It is a design pattern that can be used to obtain loose coupling between a class and its components. You can provide an object's dependencies from the outside, rather than inheriting or implementing them in the object itself. Creates flexible classes that can change their behaviour dynamically.

From the robot example above:

In [ ]:
class IndustrialRobot:
    def __init__(self, body, arm):
        self.body = body 
        self.arm = arm

    # ...

# ...

robot_ = IndustrialRobot(Body(), Arm()) # Inject appropiate object here

### 4.5 Polymorphism & Abstraction

#### **Abstraction: Creating Abstract Base Classes (ABCs) and Interfaces**

In OOP, an interface describes the **methods** a class must implement to perform a specific role.
Python has no dedicated syntax for interfaces, but it gives you two mechanisms for modeling them: abstract base classes and protocols.

An **interface** acts as a blueprint for implementing concrete classes. Interfaces define methods that are typically **abstract,** which means that the interface declares them, but does not implement them.

Sometimes, you want to create a class hierarchy in which all the classes implement a **predefined** interface or API. In other words, you want to define the specific set of public methods and attributes that all the classes in the hierarchy must implement.

##### **Inheritance-Based Interfaces with ABCs**

The **`abc`** module in Python’s **standard library** gives you several tools for working with abstract base classes (ABCs) and modeling inheritance-based interfaces.

You can use to it to define custom base classes that require all their subclasses to implement specific interfaces. A concrete class declares its intent to satisfy an interface by subclassing the corresponding ABC. You **cannot instantiate** ABCs directly. You **must subclass** them. In a sense, ABCs work as templates for other classes to inherit from.

ABCs enforce the interface contract. Any subclass that is missing a required method fails with a `TypeError` the moment you try to instantiate it.

**Defining and ABC-based Interface**

An ABC declares **abstract methods** using the `@abstractmethod` decorator, from the `abc` module.
Any subclass must implement every abstract method. Say you want to create a class hierarchy from this:


In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC): # Base class. Inherits from abc.ABC, which means it is an abst. base class.
    @abstractmethod # Common interface that all subclasses must implement
    def get_area(self): 
        pass

    @abstractmethod # Common interface that all subclasses must implement
    def get_perimeter(self):
        pass

Each abstract method is decorated with **`@abstractmethod`.** The body of abstract methods can contain a descriptive **docstring,** a bare `pass` statement, an ellipsis `(...)`, or raise a `NotImplementedError` exception.

Now, one can create concrete classes:

In [ ]:
from abc import ABC, abstractmethod
from math import pi

# ...

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def get_area(self):
        return pi * self.radius ** 2

circle = Circle(5) # Cannot instantiate. Missing implementation of the other abstract method.
                   # Must provide suitable implementation for all abstract method (define COMPLETE interface)

One will not be able to instantiate a concrete class if the common interface defined for all subclasses to implement is not actually implemented. All abstract methods must be provided an implementation.

For example:

In [ ]:
from abc import ABC, abstractmethod
from math import pi

# ...

class Circle(Shape): # NOW implements all required methods. Complete interface implementation
    def __init__(self, radius):
        self.radius = radius

    def get_area(self): # Satifies the interface
        return pi * self.radius ** 2

    def get_perimeter(self): # Satifies the interface
        return 2 * pi * self.radius

circle = Circle(10)
circle.get_area()

This demonstrates how you can use ABCs to define a **common interface** for a group of related classes. Each subclass must provide its own implementation of the abstract methods in the base class.

##### **Structural Subtyping Interfaces With Protocols**

Protocols *suggest* the interface contract rather than enforce it.

Unlike ABCs, a concrete class does not have to inherit from the target protocol. If it has the expected methods with the right signatures, then it counts as a class that implements the protocol.

Protocols are often the best fit when you do not need interface enforcement at runtime, or when the classes involved do **not** share a natural inheritance **relationship.**

**Defining a Protocol-based Interface**

In [ ]:
from typing import Protocol

class Shape(Protocol): # Inherits from typing.Protocol.
    """Protocol"""

    def calc_area(self):
        ...              # Protocols often use an ellipsis to mark protocol bodies.

    def calc_perimeter(self):
        ...

A concrete class satisfies the protocol purely by implementing the protocol’s methods:

In [ ]:
class Circle:
    def __init__(self, radius):
        self.radius = radius

    def calc_area(self):
        return 3.14 * self.radius**2

    def calc_perimeter(self):
        return 2 * 3.14 * self.radius

circle = Circle(10)

##### **Recap**

**Abstraction** means hiding implementation complexity behind a simple, well-defined interface: a caller only needs to know *what* an object can do, not *how* it does it.

**`abc.ABC` + `@abstractmethod`** turn "please override this" from a polite convention into something Python actually enforces. A class inheriting from `ABC` with one or more `@abstractmethod`s becomes an **abstract base class**: it cannot be instantiated directly, and neither can any subclass that fails to override *every* abstract method.

In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self):
        """Every concrete Shape must know how to compute its own area."""

    @abstractmethod
    def perimeter(self):
        """Every concrete Shape must know its own perimeter."""

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return 3.14159 * self.radius ** 2

    def perimeter(self):
        return 2 * 3.14159 * self.radius

try:
    Shape()                 # Can't instantiate an abstract class
except TypeError as e:
    print("Rejected:", e)

class Incomplete(Shape):
    def area(self):
        return 0
    # perimeter() is missing!

try:
    Incomplete()             # Still abstract: not every abstractmethod was overridden
except TypeError as e:
    print("Rejected:", e)

print(Circle(2).area())      # Works: Circle overrides both abstract methods

**`typing.Protocol`** offers **structural typing** (a.k.a. "static duck typing"): unlike `ABC` (*nominal* typing — a class must explicitly inherit from it to count), a `Protocol` just describes a *form* (which methods/attributes, with which signatures), and any class matches it as long as it has that form — no inheritance required. This is checked by static type checkers (mypy, pyright; not explained for now) and editors, matching how duck typing already works in practice, but adding type-checking and autocomplete support. `@runtime_checkable` additionally allows a limited `isinstance()` check against a `Protocol` at runtime.

In [ ]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class Speaker(Protocol):
    def speak(self):
        ...

class Parrot:                 # No inheritance from Speaker at all
    def speak(self):
        return "Polly wants a cracker!"

def announce(entity: Speaker):
    print(entity.speak())

announce(Parrot())                     # A type checker accepts this: Parrot's shape matches Speaker
print(isinstance(Parrot(), Speaker))   # True at runtime too, thanks to @runtime_checkable

##### **Duck Typing**

Python is well known for using *duck typing*, which is based on the idea: "if it walks like a duck and quacks like a duck, then it must be a duck."

Duck typing *relies on* the contract at runtime. In a duck typing context, completely **unrelated classes** can be used interchangeably, as long as they **expose the methods** the caller expects. Inheritance from ABC, satisfying a `Protocol` or exposing the right methods does not matter.

Duck typing is generally preferred over explicit `isinstance()` or `issubclass()` checks. **EAFP** principle: *easier to ask for forgiveness than permission*. Just call the methods you need, and let Python raise an error if the object does not support them.

Duck typing keeps your code flexible at the cost of giving up the early signal you would get from an ABC or a protocol.

**Applying Duck Typing**

In [ ]:
# No ABC Interface, no Protocol, no shared base class. Classes below have nothing in common type-wise.
# Happen to expose the same methods.

class Circle:
    """Build a circle. Calculate some parameters"""
    def __init__(self, radius):
        self.radius = radius

    def calc_area(self):
        print("Calculating circle area")
        return 3.14 * self.radius**2

    def calc_perimeter(self):
        print("Calculating circle perimeter")
        return 2 * 3.14 * self.radius

class Square:
    """Build a square. Calculate some parameters"""
    def __init__(self, side):
            self.side = side
    
    def calc_area(self):
        print("Calculating square area")
        return self.side**2
    
    def calc_perimeter(self):
        print("Calculating square perimeter")
        return 4 * self.side

def characterize_shape(shape, measure): # As long as it gets an object with both methods, it does not care which class it belong to.
    return (shape.calc_area(measure), shape.calc_perimeter(measure))

So, in duck typing the **contract** is *behaviour* rather than *type*. Python never checks what the object *is*, only what it *does* when you call a method on it.

In conclusion, Python gives you two main ways to define an interface: **abstract base classes** and **protocols.** Go for `abc.ABC` when in an inheritance context, with control over the hierarchy and need interface enforcement at instantiation. When you want solid static checking without using inheritance, choose `typing.Protocol`.

Once built an interface contract, favor **duck typing,** which relies on behaviour.

#### **Polymorphism**

Having a set of classes to implement the same interface with specific behaviors for concrete classes is a great way to unlock polymorphism.

Polymorphism is when you can use objects of different classes **interchangeably** because they share a **common interface.** For example, Python strings, lists, and tuples are all sequence data types. This means that they implement an interface that is common to all sequences.

For example:

In [ ]:
message = "text"
numbs = [1, 2, 3]
letters = ("A", "B", "C")

print(len(message),
len(numbs),
len(letters))

You can unlock polymorphism in your classes by making them **share** common **attributes and methods.** Making your classes work in a polymorphic way adds flexibility to your code.

In [ ]:
class Duck:
    def speak(self):
        return "Quack!"

class Robot:
    def speak(self):
        return "Beep boop!"

class Rock:
    pass  # no speak() at all

def make_it_speak(entity):
    return entity.speak()   # No type check: just assumes entity has .speak()

for thing in [Duck(), Robot()]:
    print(make_it_speak(thing))   # Both "quack" like ducks, despite being unrelated classes

try:
    make_it_speak(Rock())
except AttributeError as e:
    print("Failed at call-time, not before:", e)

So, in the end:

**Polymorphism** ("many forms") means code written against a general interface can work with any object that implements that interface, regardless of its concrete type. Python supports two flavors:

1. **Inheritance-based** — subclasses override a common method declared in a shared base class.
2. **Duck typing** (dynamic) — "if it walks like a duck and quacks like a duck, it's a duck." Python does not check types before calling a method; it just tries, and fails at *call time* (`AttributeError`) if unsupported. No inheritance relationship is required between unrelated classes that happen to share a method name.

### 4.6 Composition vs Inheritance

Inheritance and composition are two major concepts in object-oriented programming that model the relationship between two classes. Both of them enable code reuse, but they do it in different ways.

#### **Inheritance**

Inheritance models what is called an **is a** relationship. This means that when you have a Derived class that inherits from a Base class, you have created a relationship where `Derived` is a specialized version of `Base`.

The inheritance relationship comprises that `Derived` inherits the **interface** and implementation of `Base`, and you can use `Derived` objects to replace `Base` objects in the application. 

This is called the **Liskov Substitution principle:** *if S is a subtype of T, then replacing objects of type T with objects of type S does not change the program’s behavior.*


#### **Composition**

Composition models what is called a **has a** relationship. It enables creating complex types by combining objects of other types. This means that a class `Composite` can contain an object of another class `Component`. `Composite` has a `Component`.

Composition enables you to reuse code by adding objects to other objects, as opposed to inheriting the interface and implementation of other classes. 

#### **Inheritance Overview**

Everything in Python is an **object.** Modules are objects, class definitions and functions are objects, and of course, objects created from classes are objects too.

Inheritance is a required feature of every OOP language. This means that Python supports inheritance, and it is one of the few that supports **multiple** inheritance.

When writing a simple class, one is already using inheritance even if it does not look like.

##### **The Object Super Class**

Let us declare the simplest class possible:

In [ ]:
class EmptyClass:
    pass

... and create an instance and use `dir()` function to list its members:

In [ ]:
c = EmptyClass()
dir(c)

...but no members were declared in the class above. The list actually comes from:

In [ ]:
o = object()
dir(o)

Although, the class above has three additional members: `__dict__`, `__module__` and `__weakref__`. However, every single member of the `object` class is also present in our class. 

This is because every class that you create in Python implicitly **derives from `object`**. But:

##### **Exceptions are an Exception**

Every class that you create in Python will implicitly derive from object. However, there is one exception to this rule: classes used to indicate errors by raising an exception. For example, if you create a class and `raise` it as an exception:

In [ ]:
class NotAnError:
    pass

raise NotAnError()

Python does  raise an exception, but the output states that the exception is of type `TypeError`, and that all exceptions must derive from `BaseException`.

**`BaseException`** is the base class provided for all error types. To create a new error type, you must derive your class from BaseException or one of its derived classes. The convention is to derive your custom error type from **`Exception`**, which derives from `BaseException`:

In [ ]:
class AnError(Exception):
    pass

raise AnError()

##### **Creating Class Hierarchies**

Inheritance is the mechanism that one will use to create **hierarchies** of related classes. These related classes will share a **common interface** that the base classes will define. Derived classes can specialize the interface by providing a particular implementation where applicable. 

#### **Composition in Python**

Composition is an OOP design concept that models a **has a** relationship. In composition, a class known as **composite** contains an object, or **component,** of another class. 

Composition allows composite classes to reuse the implementation of the components it contains. The composite class does not inherit the component class interface.

If classes are **related** by composition, they are considered **loosely coupled.** That means that changes to the component class rarely affect the composite class, and changes to the composite class never affect the component class.

When looking at two competing software designs, one based on inheritance and another based on composition, the composition solution usually is more **flexible.**

In fact, looking at a simple class, one can realize to have used already composition by assigning attributes during instantiation, since they are also objects:

In [ ]:
class Composite:
    def __init__(self, attr1, attr2):
        self.attr1 = attr1
        self.attr2 = attr2

# You can say that this class **has a** .attr1 and **has a** attr2, which are both objects.

comp = Composite("a", "b")

Composition is a loosely coupled relationship that often does not require the composite class to have knowledge of the component.

##### **Flexible Designs with Composition**

Composition is more flexible than inheritance because it models a loosely coupled relationship. Changes to a component class have minimal or no effects on the composite class. Designs based on composition are more suitable to change.

You change behavior by providing new components that implement those behaviors instead of adding new classes to your hierarchy.

For example:

In [ ]:
from math import sqrt

class DistanceComp:
    def __init__(self, p, q):
        self.p = Point(1, 2)
        self.q = Point(5, 8)

    def distance_calc(self):
        return sqrt((self.p.x - self.q.x)**2 + (self.p.y - self.q.y)**2)

class PointComp:
    def __init__(self, x, y):
        self.x = x
        self.y = y

distance = DistanceComp(PointComp, PointComp)
distance.distance_calc()

# I can freely change the Point classes which my Distance class is **composed** of.

...which is more flexible than:

In [ ]:
from math import sqrt

class PointInh:
    def __init__(self, x, y):
        self.x = x
        self.y = y

class DistanceInh(PointInh): # Changes to Point classes have a greater effect on the derived class Distance.
    def __init__(self, p, q):
        self.p = p
        self.q = q

    def distance_calc(self):
        return sqrt((self.p.x - self.q.x)**2 + (self.p.y - self.q.y)**2)

distance = DistanceInh(PointInh(1, 2), PointInh(3, 5))
distance.distance_calc()

#### **Choosing Between Inheritance and Composition**

Inheritance and composition can tackle similar problems in your Python programs. They both enable code **reuse.**

The general advice is to use the relationship that creates **fewer dependencies** between two classes. This relation is composition. But sometimes inheritance can make more sense.

##### **Inheritance to model an "Is A" Relationship**

You should only use inheritance to model an **is a** relationship. **Liskov’s** principle says that an object of type `Derived`, can replace an object of type `Base` without altering the desirable properties of a program.

Liskov’s substitution principle is the most important guideline to determine if inheritance is the appropriate design solution. Let’s say you have a class, A, that provides an implementation and interface you want to reuse in another class, B. An initial thought can be that you can derive B from A and inherit interf. + implementation. To be sure:

* **Evaluate** B is an A.
* **Evaluate** A is an B.

If both cases can be justified, then you should never inherit those classes from one another. For example: 

You have a `Rectangle` class that exposes an `.area` property. You need a `Square` class, which also has an `.area`. It seems that a `Square` is a special type of Rectangle, so maybe you can derive from it and leverage both the interface and implementation:

In [ ]:
class Rectangle:
    def __init__(self, length, height):
        self._length = length
        self._height = height

    @property
    def area(self):
        return self._length * self._height

In [ ]:
class Square(Rectangle):
    def __init__(self, side_size):
        super().__init__(side_size, side_size)

In [ ]:
rectangle1 = Rectangle(1, 3)
square1 = Square(2)
print(rectangle1.area, square1.area)

Looks like everything works fine, but imagine some changes need to be done to `Rectangle`:

In [ ]:
class Rectangle:
    def __init__(self, length, height):
        self._length = length
        self._height = height

    @property
    def area(self):
        return self._length * self._height

    def resize(self, new_length, new_height):
        self._length = new_length
        self._height = new_height

class Square(Rectangle):
    def __init__(self, side_size):
        super().__init__(side_size, side_size)

In [ ]:
rectangle1 = Rectangle(1, 3)
rectangle1.resize(2, 4)
rectangle1.area

square1 = Square(2)
square1.resize(2, 4)
square1.area

Our `Square` object literally broke apart, since it calculated its area as if it was a `Rectangle` object, getting resized. In the example, it does noy make sense that Square inherits the interface and implementation of `.resize()` from Rectangle. That doesn’t mean that Square objects cannot be resized. It means that the interface is different because it only needs a `side_size` parameter.

The reality is that if you are able to justify an inheritance relationship between two classes both ways, then you should not derive one class from another.

##### **Composition to Model “Has A” Relationship**

With composition, a class `Composite` has an instance of the class `Component` and can leverage its implementation. You can **reuse** the `Component` class in other classes completely **unrelated** to the `Composite`.

A problem that you may run into when using composition is that some of your classes may start growing by using multiple components. Your classes may require multiple parameters in the constructor just to pass in the components that they’re made of. This can make your classes hard to use.

Inheritance, as opposed to composition, is a tightly coupled relationship. With inheritance, there is only one way to change and customize behavior. **Method overriding** is the only way to customize the behavior of a base class. This creates rigid designs that are difficult to change.

Composition, on the other hand, provides a loosely coupled relationship that enables flexible designs and can be used to change behavior at **runtime.**

##### **Some Guidelines**

It can be hard to see what the relationships between some classes can be, but one can follow these guidelines: 

* **Use inheritance over composition** to model a clear **is a** relationship. Only if you can’t justify the relationship in both directions should you use inheritance between them.
* **Use inheritance over composition** to leverage both the **interface and implementation** of the base class.
* **Use composition over inheritance** to model a **has a** relationship that leverages the implementation of the component class.
* **Use composition over inheritance** to create **components** that multiple classes in your Python applications can reuse.
* **Use composition over inheritance** to implement **groups of behaviors** and policies that can be applied interchangeably to other classes to customize their behavior.
* **Use composition over inheritance**  to enable **runtime** behavior changes without affecting existing classes.

#### **Conclusion**

Inheritance models an **"is-a"** relationship (`Cat` **is a(n)** `Animal`). **Composition** models a **"has-a"** relationship, by storing another object as an attribute and delegating to it (`Car` **has a(n)** `Engine`).

Overusing inheritance causes real problems: the **"fragile base class" problem** (changing a base class can unexpectedly break subclasses relying on its internal details), rigid deep hierarchies (which of these ancestors actually defines this method?), and the fact that inheritance is fixed at class-definition time, while composition lets you swap a component object at runtime. "Inheriting for code reuse" alone — when `B` isn't conceptually a *kind of* `A`, but inherits from it just to reuse a method — is a common design smell.

**Guideline:** reach for inheritance only when there's a genuine is-a/substitutability relationship (a `Cat` really can be used anywhere an `Animal` is expected — the Liskov Substitution Principle). Reach for composition whenever you just want to reuse or combine behavior. In practice, professional Python code favors composition far more than beginners expect.

In [ ]:
class Engine:
    def start(self):
        return "Engine roaring to life..."

class ElectricEngine:
    def start(self):
        return "Silent electric hum..."

class Car:
    def __init__(self, engine):
        self.engine = engine   # "has-a" Engine: composition, not inheritance

    def start(self):
        return self.engine.start()   # delegates to the composed object

gas_car = Car(Engine())
electric_car = Car(ElectricEngine())

print(gas_car.start())
print(electric_car.start())

# Composition also allows swapping the component at runtime, which inheritance can't do:
gas_car.engine = ElectricEngine()
print(gas_car.start())   # Same Car object, new behavior, no new class needed

### 4.7 Dataclasses

#### **Introduction**

A **data class** is a class typically containing mainly **data,** although there are not really any restrictions. To create a dataclass, use the **`@dataclass`** decorator:

```python 
from dataclassses import dataclass

@dataclass
class ExampleDataclass:
    data1: int
    data2: str
```

Basic functionality is this type of classes is already implemented: instantiation, printing, data comparison. Compare the syntax of a standard dataclass like before with the one of a regular class:

```python
class ExampleClass:
    def __init__(self, attr1, attr2):
        self.attr1 = attr1
        self.attr2 = attr2
    ...
```

Definition and boilerplate start to get repetitive, the representation of objects may not be very descriptive and basic object comparisons may not return expected results:

In [ ]:
class ExampleClass:
    def __init__(self, attr1, attr2):
        self.attr1 = attr1
        self.attr2 = attr2
    ...

example = ExampleClass(1, 2)
print(example == ExampleClass(1, 2)) # Technically, it returns False, as it is expected, but for practical purposes we should expect a True

#### **Basic Data Classes**

What makes a class a dataclass is, as said before, the `@dataclass` decorator just above the class definition. Below the class definition, the **fields** that one wants in the dataclass are listed. The "`:`" notation used for the fields is using a feature called **variable annotations.** 

For example:

In [ ]:
from dataclasses import dataclass

@dataclass
class Point:
    name: str
    x: float
    y: float

These lines are enough for a definition. The class is ready to use:

In [ ]:
point = Point("A", 1, 2)
print(point)

The following is also equivalent to the definition above:

In [ ]:
from dataclasses import make_dataclass

Point = make_dataclass("Point", ["name", "x", "y"])

A data class is a regular Python class. The only thing that sets it apart is that it has basic data model methods like `.__init__()`, `.__repr__()`, and `.__eq__()` implemented for you.

##### **Default Values**

To add default values to the fields of a dataclass:

In [ ]:
from dataclasses import dataclass

@dataclass
class Point:
    name: str
    x: float = 0.0
    y: float = 0.0

This works exactly as if you had specified the default values in the definition of the `.__init__()` method of a regular class:

In [ ]:
print(Point("A"))
print(Point("A", 1))
print(Point("A", 1, 2))

##### **Type Hints**

Dataclasses support **typing** directly. When defining the fields, type hints were used, indicating the `type` each fields should have. 

Adding some kind of type hint is mandatory when defining the fields in your data class. **Without** a type hint, the field will **not** be a part of the data class.

However, if you do not want to add explicit types to your data class, use `typing.Any`:

In [ ]:
from dataclasses import dataclass
from typing import Any

@dataclass
class WithoutExplicitTypes:
    name: Any
    value: Any = 0

While you need to add type hints in some form when using data classes, these types are not enforced at runtime. For example:

In [ ]:
Point(3, "B", True) # Types were not enforced. No sense, but it is possible.

##### **Adding Methods**

As a regular class, methods be freely added and implemented. For example:

In [ ]:
from dataclasses import dataclass
from math import sqrt

@dataclass
class Point:
    name: str
    x: float = 0.0
    y: float = 0.0

    def distance_origin(self):
        return sqrt(self.x**2 + self.y**2)

#### **More Flexible Dataclasses**

##### **Advanced Default Values**

Data classes use something called a `default_factory` to handle mutable default values. Using **mutable default** arguments is a common anti-pattern in Python. For example:

In [ ]:
from dataclasses import dataclass

@dataclass
class A:
    x: list = []
    def add(self, element):
        self.x.append(element)


# All instances of A will use the same list object as default value for x. If something
# is added or removed from one instance, it will affect others:
a1 = A([1])
print(a1)

a2 = A([2])
print(a2)

# In fact, you cannot expect to run the code above. Dataclasses warn you.

Actually, data classes try to prevent you from doing this, and the code above will raise a `ValueError`. The dataclass from above is roughly equivalent to this class:

In [ ]:
class A:
    x = []    
    def __init__(self, x=x):
        self.x = x
    def add(self, element):
        self.x.append(element)

To solve this problem, use **`default_factory`** to handle mutable default values, for which you need to use the **`field()`** specifier:

In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class B:
    y: list = field(default_factory=list)
    def add(self, element):
        self.y.append(element)

b = B()
b.add(1)
print(b)

The argument to `default_factory` **must** be any **zero** parameter callable (`list`, `dict`...). The `field()` specifier is used to customize each field of a data class individually. This is every parameter `field()` supports:

* `default`: dafualt value of the field.
* `default_factory`: function that returns the initial value of the field.
* `init`: use field in `__init__()` method? `True` by default.
* `repr`: use field in the object's `repr`? `True` by default.
* `compare`: include field in comparisons? `True` by default.
* `init`: include field when calculatin `hash`? Default is to use the same as `compare`.
* `metadata`: a mapping with information about the field.

The metadata parameter is not used by the data classes themselves but is available for you, to attach information to fields. For example:

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Position:
    name: str
    x: float = field(default=0.0, metadata={"unit": "m"})
    y: float = field(default=0.0, metadata={"unit": "m"})

Any information about a field can be retrieved using the `fields()` function:

In [ ]:
from dataclasses import fields
fields(Position)

In [ ]:
y_unit = fields(Position)[2].metadata['unit']
print(y_unit)

##### **Representation**

In general, a Python object has two different string representations, as seen previously:

* `repr(obj)` is defined by `obj.__repr__()` and should return a developer-friendly representation of obj. If possible, this should be code that can recreate `obj`. Data classes do this.
* `str(obj)` is defined by `obj.__str__()` and should return a user-friendly representation of obj. Data classes do not implement a `.__str__()` method, so Python will fall back to the `.__repr__()` method.

You can also add a user-friendly representation to a dataclass:

In [ ]:
from dataclasses import dataclass

@dataclass
class Position:
    name: str
    x: float
    y: float 

    def __str__(self):
        return f"Position {self.name}: (x = {self.x}, y = {self.y})"

print(Position("A", 1, 2))

However, it is also possible to add your own `.__repr__()` method as well even if `dataclass` already implements it.

##### **Comparisons**

The `@dataclass` decorator has two forms. The simple form is where `@dataclass` is specified without any parentheses and parameters. But parameters can be given to this decorator; the following are supported:

* `init`: add `.__init__()` method? (Default is `True`.)
* `repr`: add `.__repr__()` method? (Default is `True`.)
* `eq`: add `.__eq__()` method? (Default is `True`.)
* `order`: add ordering methods? (Default is `False`.)
* `unsafe_hash`: force the addition of a `.__hash__()` method? (Default is `False`.)
* `frozen`: if `True`, assigning to fields raise an exception. (Default is `False`.)

(For more information, visit: [dataclasses - Python 3 Documentation](https://docs.python.org/3/library/dataclasses.html#) or [PEP - Dataclasses](https://peps.python.org/pep-0557/#id7)). 

Dataclasses compare objects as if they were **tuples** of their fields:

```python
>>> (field1A, field2A, ...) > (field1B, field2B, ...)
True # Or False, boolean
```

... which may lead to incorrect comparisons if fields contain other classes of data that are not numeric. However, there is a special method that comes in handy: **`__post_init__`**. It allows for special processing after the regular `__init__` method is called. For example, let us create a dataclass of playing cards, implementing successful ordering:

In [ ]:
from dataclasses import dataclass, field

RANKS = '2 3 4 5 6 7 8 9 10 J Q K A'.split()
SUITS = '♣ ♢ ♡ ♠'.split()

@dataclass(order=True) # After setting order=True, we allow for comparisons between instances.
class PlayingCard:
    sort_index: int = field(init=False, repr=False) # First field. That way the comparison is first
                                                    # done using sort_index. Also, must specify to 
                                                    # not be included in init
    rank: str
    suit: str

    def __post_init__(self):
        self.sort_index = (RANKS.index(self.rank) * len(SUITS)
                           + SUITS.index(self.suit))

    def __str__(self):
        return f'{self.suit}{self.rank}'

ace_spades = PlayingCard("A", "♠")
queen_hearts = PlayingCard('Q', '♡')
ace_spades > queen_hearts # True, as should be. Compare this result to the one removing sort_index and post_init.

##### **Immutable Dataclasses**

For some types of dataclasses, it is a good idea to make the values of the fields to **not** change. To make a dataclass **immutable**, include **`frozen=True`** when creating it. For example, an immutable version of one of the classes defined above:

In [ ]:
from dataclasses import dataclass, field

@dataclass(frozen=True)
class ImmutablePoint:
    name: str
    x: float = 0.0
    y: float = 0.0

point = ImmutablePoint("C", 3, 4)
print(point.name)
point.name = "D" # FrozenInstanceError. We specified our dataclass as immutable.

One will get a `FrozenInstanceError` if trying to assign a value to a field after "freezing" the class. Even if declaring the dataclass immutable, if any field contains a mutable data type, it may still be able to change.

In [ ]:
from dataclasses import dataclass, field

@dataclass(frozen=True)
class ImmutableList:
    l: list = field(default_factory=list)

listclass = ImmutableList([1, 2, 3])
print(listclass)

listclass.l[0] = 4
print(listclass)

##### **Inheritance**

Of course, dataclasses can be **subclassed** following inheritance relationships. For example:

In [ ]:
from dataclasses import dataclass

@dataclass
class Position:
    name: str
    lon: float
    lat: float

@dataclass
class Capital(Position):
    country: str

Capital("Madrid", 1.0, 2.0, "Spain")

The fields of the subclass(es) are added **after** the original fields of the base class(es). However, if these last have default values:

In [ ]:
from dataclasses import dataclass

@dataclass
class Position:
    name: str
    lon: float = 0.0
    lat: float = 0.0

@dataclass
class Capital(Position):
    country: str  # Does NOT work

...results in a `TypeError`: non-default argument {...} follows default argument. This is because the dataclass tried to write and `init` method of the form:

```python
def __init__(arg1: str, arg2: float = 0.0, ... argn: str):
    ...
```
...which is **not** valid. If a parameter has a [default value](#Default_Arguments), all following parameters must also have a default value. In other words, if a(ny) field in a base class has a default value, then all new fields added in a subclass must have default values as well.

Another thing to be aware of is how fields are ordered in a subclass. Starting with the base class, fields are ordered in the order in which they are first defined. If a field is redefined in a subclass, its order does not change.

In [ ]:
from dataclasses import dataclass

@dataclass
class Position:
    name: str
    lon: float = 0.0
    lat: float = 0.0

@dataclass
class Capital(Position):
    country: str = "Unknown"
    lat: float = 10.0

Capital("Madrid", country = "Spain")

##### **Optimizing Dataclasses**

[**Slots**](#lightweight-classes-slots) can be used to make classes faster and use less memory. Data classes have no explicit syntax for working with slots, but the normal way of creating slots works for data classes as well:

In [ ]:
from dataclasses import dataclass

@dataclass
class SlotPoint:
    __slots__ = ["name", "x", "y"]
    name: str
    x: float
    y: float

Variables or attributes not present in `.__slots__` should not be defined. Furthermore, a slots class may not have default values. 

The benefits of using slots as restrictions is that optimizations can be done. Slots classes take up less memory and are faster to work with.

##### **Conclusion**

Plain classes meant to just hold a handful of related values require repetitive boilerplate: an `__init__` that assigns every field by hand, a `__repr__` for readable printing, and an `__eq__` for value comparison — without them you get identity comparison and an unhelpful default repr.

**`@dataclass`** (standard library, `dataclasses` module) auto-generates `__init__`, `__repr__`, and `__eq__` from **type-annotated class attributes**, called *fields*. It's still just a regular class underneath — you can add ordinary methods too; `@dataclass` only affects what gets auto-generated.

**`field()`** is used when a field needs more than a plain default value — most importantly `field(default_factory=list)` for a mutable default. Dataclasses explicitly forbid writing `tags: list = []` directly (to prevent exactly the shared-mutable-default trap from section 4.1); `default_factory` instead calls a zero-argument function to build a *fresh* object for every new instance.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Point:
    x: float
    y: float
    tags: list = field(default_factory=list)   # avoids the mutable-default trap from section 4.1

p1 = Point(1, 2)
p2 = Point(1, 2)

print(p1)                # auto-generated __repr__
print(p1 == p2)          # auto-generated __eq__: compares field values, not identity
p1.tags.append("origin-ish")
print(p1.tags, p2.tags)  # each instance got its own list

**`frozen=True`** makes instances immutable after construction: any attempt to assign to a field afterward raises `FrozenInstanceError`. It also makes the generated `__hash__` usable — a frozen dataclass (with `eq=True`, the default) is hashable, because immutability guarantees equal objects will keep the same hash forever. This is exactly the "only hash immutable objects" rule from section 4.3, applied automatically. Frozen dataclasses are a natural fit for value objects that shouldn't change (coordinates, money amounts, config).

**`__post_init__`** is a hook that runs automatically right after the generated `__init__` finishes. Since `@dataclass` writes `__init__` for you, this is where you add validation or compute derived fields that depend on the given ones.

In [ ]:
@dataclass(frozen=True)
class Coordinates:
    lat: float
    lon: float

    def __post_init__(self):
        if not (-90 <= self.lat <= 90):
            raise ValueError("Latitude out of range")

home = Coordinates(40.4, -3.7)
print(home)
print({home, Coordinates(40.4, -3.7)})   # frozen + eq -> hashable; these two are equal, so the set collapses them

try:
    home.lat = 0              # frozen: can't mutate after construction
except Exception as e:
    print("Rejected:", type(e).__name__, e)

try:
    Coordinates(999, 0)        # __post_init__ validation runs right after the fields are set
except ValueError as e:
    print("Rejected:", e)

For more information about dataclasses: [PEP 557](https://peps.python.org/pep-0557/)